In [1]:
import os, pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = '/Users/tm/Documents/GitHub/Project-for-Work/data/processed'
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
TRAIN_NEG_RATIO = 3
BATCH_SIZE  = 512
RANDOM_SEED = 42

feature_matrix = pd.read_csv('/Users/tm/Documents/GitHub/Project-for-Work/data/processed/feature_matrix.csv')
with open('/Users/tm/Documents/GitHub/Project-for-Work/data/processed/feature_cols.pkl', 'rb') as _f:
    feature_cols = pickle.load(_f)
print(f'feature_matrix: {feature_matrix.shape} | features: {len(feature_cols)}')

feature_df = feature_matrix  # alias


feature_matrix: (147063, 21) | features: 17


---
## Stage 10: Dataset Building & Entity-Aware Split

**วัตถุประสงค์:** แบ่ง data เป็น train/val/test อย่างถูกต้อง (ไม่มี data leakage)

**⚠️ กฎสำคัญ:**
- ❌ ห้าม random split pairs → entity เดียวกันจะอยู่ทั้ง train+test (leakage!)
- ✅ แบ่ง **entities** ก่อน → pairs ตามไป
- ✅ `scaler.fit(train)` เท่านั้น → `transform(val/test)`

| Sub-step | หน้าที่ |
|----------|--------|
| 10.1 | Entity-Aware Split (70/15/15) |
| 10.2 | Class Imbalance Handling |
| 10.3 | Feature Scaling (fit on train only!) |
| 10.4 | สร้าง PyTorch DataLoaders |

### Step 10.1: Entity-Aware Train/Val/Test Split
แบ่ง **entities** (ไม่ใช่ pairs) → ป้องกัน data leakage

In [2]:
# --- 10.1 Entity-Aware Split ---
from sklearn.preprocessing import StandardScaler

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# แบ่ง entities (ไม่ใช่ pairs!)
unique_entities = feature_df['entity_id_a'].unique()
np.random.seed(RANDOM_SEED)
np.random.shuffle(unique_entities)

n = len(unique_entities)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * (TRAIN_RATIO + VAL_RATIO))

train_entities = set(unique_entities[:n_train])
val_entities = set(unique_entities[n_train:n_val])
test_entities = set(unique_entities[n_val:])

# Filter pairs ตาม entity group
train_df = feature_df[feature_df['entity_id_a'].isin(train_entities)].copy()
val_df = feature_df[feature_df['entity_id_a'].isin(val_entities)].copy()
test_df = feature_df[feature_df['entity_id_a'].isin(test_entities)].copy()

print("📊 Step 10.1: Entity-Aware Split")
print("=" * 60)
print(f"  Total entities : {n:,}")
print(f"  Train entities : {len(train_entities):,}")
print(f"  Val entities   : {len(val_entities):,}")
print(f"  Test entities  : {len(test_entities):,}")
print(f"\n  Train pairs: {len(train_df):,} (pos={train_df['label'].sum():.0f}, neg={len(train_df)-train_df['label'].sum():.0f})")
print(f"  Val pairs  : {len(val_df):,} (pos={val_df['label'].sum():.0f}, neg={len(val_df)-val_df['label'].sum():.0f})")
print(f"  Test pairs : {len(test_df):,} (pos={test_df['label'].sum():.0f}, neg={len(test_df)-test_df['label'].sum():.0f})")

# ✅ ตรวจสอบ leakage
overlap_tv = train_entities & val_entities
overlap_tt = train_entities & test_entities
overlap_vt = val_entities & test_entities
print(f"\n  ✅ Leakage check:")
print(f"    Train∩Val  = {len(overlap_tv)} {'✅' if len(overlap_tv)==0 else '❌ LEAKAGE!'}")
print(f"    Train∩Test = {len(overlap_tt)} {'✅' if len(overlap_tt)==0 else '❌ LEAKAGE!'}")
print(f"    Val∩Test   = {len(overlap_vt)} {'✅' if len(overlap_vt)==0 else '❌ LEAKAGE!'}")
print(f"\n✅ Step 10.1 เสร็จ")

📊 Step 10.1: Entity-Aware Split
  Total entities : 9,124
  Train entities : 6,386
  Val entities   : 1,369
  Test entities  : 1,369

  Train pairs: 103,159 (pos=16313, neg=86846)
  Val pairs  : 21,967 (pos=3452, neg=18515)
  Test pairs : 21,937 (pos=3441, neg=18496)

  ✅ Leakage check:
    Train∩Val  = 0 ✅
    Train∩Test = 0 ✅
    Val∩Test   = 0 ✅

✅ Step 10.1 เสร็จ


### Step 10.2: Class Imbalance Handling
Undersample negatives ใน train set / คำนวณ class weights

In [3]:
# --- 10.2 Class Imbalance ---
TRAIN_NEG_RATIO = 3  # ratio ใน training set หลัง undersample

train_pos = train_df[train_df['label'] == 1]
train_neg = train_df[train_df['label'] == 0]
n_target_neg = len(train_pos) * TRAIN_NEG_RATIO

if len(train_neg) > n_target_neg:
    train_neg_sampled = train_neg.sample(n=n_target_neg, random_state=RANDOM_SEED)
    train_balanced = pd.concat([train_pos, train_neg_sampled]).sample(frac=1, random_state=RANDOM_SEED)
else:
    train_balanced = train_df.copy()

# Class weights สำหรับ loss function
n_pos_b = (train_balanced['label'] == 1).sum()
n_neg_b = (train_balanced['label'] == 0).sum()
pos_weight = n_neg_b / max(n_pos_b, 1)

print("📊 Step 10.2: Class Imbalance Handling")
print("=" * 60)
print(f"  Before: pos={len(train_pos):,} neg={len(train_neg):,} ratio={len(train_neg)/max(len(train_pos),1):.1f}:1")
print(f"  After : pos={n_pos_b:,} neg={n_neg_b:,} ratio={n_neg_b/max(n_pos_b,1):.1f}:1")
print(f"  pos_weight = {pos_weight:.3f} (สำหรับ BCEWithLogitsLoss)")
print(f"\n✅ Step 10.2 เสร็จ")

📊 Step 10.2: Class Imbalance Handling
  Before: pos=16,313 neg=86,846 ratio=5.3:1
  After : pos=16,313 neg=48,939 ratio=3.0:1
  pos_weight = 3.000 (สำหรับ BCEWithLogitsLoss)

✅ Step 10.2 เสร็จ


### Step 10.3: Feature Scaling
⚠️ `scaler.fit()` บน train เท่านั้น → `transform()` val+test

In [4]:
# --- 10.3 Feature Scaling ---
scaler = StandardScaler()

# ⚠️ FIT ON TRAIN ONLY!
X_train = scaler.fit_transform(train_balanced[feature_cols].values)
y_train = train_balanced['label'].values.astype(np.float32)

X_val = scaler.transform(val_df[feature_cols].values)
y_val = val_df['label'].values.astype(np.float32)

X_test = scaler.transform(test_df[feature_cols].values)
y_test = test_df['label'].values.astype(np.float32)

# Save scaler
scaler_path = '/Users/tm/Documents/GitHub/Project-for-Work/data/processed/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print("📊 Step 10.3: Feature Scaling")
print("=" * 60)
print(f"  ⚠️ scaler.fit() บน train set เท่านั้น ({len(X_train):,} samples)")
print(f"  X_train: {X_train.shape} mean≈{X_train.mean():.4f} std≈{X_train.std():.4f}")
print(f"  X_val  : {X_val.shape}")
print(f"  X_test : {X_test.shape}")
print(f"  💾 Saved: scaler.pkl")
print(f"\n✅ Step 10.3 เสร็จ")

📊 Step 10.3: Feature Scaling
  ⚠️ scaler.fit() บน train set เท่านั้น (65,252 samples)
  X_train: (65252, 17) mean≈-0.0000 std≈0.9701
  X_val  : (21967, 17)
  X_test : (21937, 17)
  💾 Saved: scaler.pkl

✅ Step 10.3 เสร็จ


### Step 10.4: สร้าง PyTorch DataLoaders

In [6]:
# --- 10.4 PyTorch DataLoaders ---
import torch
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    """Dataset สำหรับ feature-based pair classification"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BATCH_SIZE = 512

train_dataset = PairDataset(X_train, y_train)
val_dataset = PairDataset(X_val, y_val)
test_dataset = PairDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# ตรวจสอบ batch
sample_X, sample_y = next(iter(train_loader))

print("=" * 60)
print("📊 STAGE 10 SUMMARY — Dataset Building")
print("=" * 60)
print(f"  Train loader : {len(train_loader)} batches ({len(train_dataset):,} samples)")
print(f"  Val loader   : {len(val_loader)} batches ({len(val_dataset):,} samples)")
print(f"  Test loader  : {len(test_loader)} batches ({len(test_dataset):,} samples)")
print(f"  Batch shape  : X={sample_X.shape}, y={sample_y.shape}")
print(f"  Input dim    : {sample_X.shape[1]} features")
print(f"  pos_weight   : {pos_weight:.3f}")
print(f"\n{'='*60}")
print(f"✅ Stage 10 COMPLETE — DataLoaders พร้อมสำหรับ Training")
print(f"{'='*60}")

📊 STAGE 10 SUMMARY — Dataset Building
  Train loader : 128 batches (65,252 samples)
  Val loader   : 43 batches (21,967 samples)
  Test loader  : 43 batches (21,937 samples)
  Batch shape  : X=torch.Size([512, 17]), y=torch.Size([512])
  Input dim    : 17 features
  pos_weight   : 3.000

✅ Stage 10 COMPLETE — DataLoaders พร้อมสำหรับ Training
